# DUMMY_DATA_SETUP
```
=============================================================================
dbx_retail — DUMMY DATA SETUP SCRIPT
=============================================================================
Creates all Gold and Silver tables required to run UC1, UC2, UC3, UC4.
Run this notebook ONCE before running any UC notebook.

Tables created:
  SILVER
    dbx_retail.silver.quarantine_customers
    dbx_retail.silver.quarantine_orders
    dbx_retail.silver.quarantine_transactions

  GOLD — Dimensions
    dbx_retail.gold.dim_customers
    dbx_retail.gold.dim_vendors
    dbx_retail.gold.dim_products

  GOLD — Facts
    dbx_retail.gold.fact_orders
    dbx_retail.gold.fact_returns

  GOLD — Materialized Views (aggregations)
    dbx_retail.gold.mv_revenue_by_region
    dbx_retail.gold.mv_return_rate_by_vendor
    dbx_retail.gold.mv_slow_moving_products

Designed from real dbx_retail source file schemas:
  customers_1.csv  : customer_id, customer_email, customer_name, segment,
                     country, city, state, postal_code, region
  orders_1.csv     : order_id, customer_id, vendor_id, ship_mode,
                     order_status, order_purchase_date, ...
  transactions_1.csv: Order_id, Product_id, Sales, Quantity, discount,
                      profit, payment_type, payment_installments
  returns_1.json   : order_id, return_reason, return_date,
                     refund_amount, return_status
  products.json    : product_id, categories, brand, product_name, ...
  vendors.csv      : vendor_id, vendor_name
=============================================================================
```

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS dbx_retail.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS dbx_retail.gold")

print("Schemas ready: dbx_retail.silver | dbx_retail.gold")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, FloatType, DoubleType
)
from datetime import datetime

print("Imports done.")


# =============================================================================
# SILVER LAYER — Quarantine tables (UC1 reads these)
# =============================================================================

In [0]:
# Simulates records rejected during Silver harmonization.
# Contains a mix of CRITICAL (missing ID), HIGH (invalid reference),
# and MEDIUM (format) issues to make UC1 severity tiering visible.

quarantine_customers_data = [
    # issue_type,           field_affected,   quarantine_timestamp,  source_file,     raw_value,       region
    ("missing_value",       "customer_id",    "2024-01-15 02:10:00", "customers_3.csv", "NULL",         "West"),
    ("missing_value",       "customer_id",    "2024-01-15 02:10:00", "customers_3.csv", "NULL",         "West"),
    ("missing_value",       "customer_id",    "2024-01-15 02:10:00", "customers_5.csv", "NULL",         "South"),
    ("missing_value",       "customer_id",    "2024-01-15 02:10:00", "customers_5.csv", "NULL",         "South"),
    ("missing_value",       "customer_id",    "2024-01-15 02:10:00", "customers_5.csv", "NULL",         "North"),
    ("invalid_segment",     "segment",        "2024-01-15 02:10:00", "customers_2.csv", "UNKNWN",       "East"),
    ("invalid_segment",     "segment",        "2024-01-15 02:10:00", "customers_2.csv", "N/A",          "East"),
    ("invalid_segment",     "segment",        "2024-01-15 02:10:00", "customers_4.csv", "Unassigned",   "Central"),
    ("missing_value",       "customer_email", "2024-01-15 02:10:00", "customers_6.csv", "NULL",         "West"),
    ("missing_value",       "customer_email", "2024-01-15 02:10:00", "customers_6.csv", "NULL",         "West"),
    ("missing_value",       "customer_email", "2024-01-15 02:10:00", "customers_6.csv", "NULL",         "East"),
    ("duplicate_record",    "customer_id",    "2024-01-15 02:10:00", "customers_1.csv", "AB-10060",     "East"),
    ("duplicate_record",    "customer_id",    "2024-01-15 02:10:00", "customers_2.csv", "AB-10060",     "East"),
    ("invalid_format",      "postal_code",    "2024-01-15 02:10:00", "customers_3.csv", "ABCDE",        "West"),
    ("invalid_format",      "postal_code",    "2024-01-15 02:10:00", "customers_4.csv", "00000",        "North"),
]

quarantine_customers_schema = StructType([
    StructField("issue_type",            StringType(), True),
    StructField("field_affected",        StringType(), True),
    StructField("quarantine_timestamp",  StringType(), True),
    StructField("source_file",           StringType(), True),
    StructField("raw_value",             StringType(), True),
    StructField("region",                StringType(), True),
])

df_qc = spark.createDataFrame(quarantine_customers_data, schema=quarantine_customers_schema)
df_qc.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
     .saveAsTable("dbx_retail.silver.quarantine_customers")

print(f"quarantine_customers : {df_qc.count()} rows")
display(df_qc)

In [0]:

quarantine_orders_data = [
    ("missing_value",       "customer_id",       "2024-01-15 02:20:00", "orders_2.csv",  "NULL",              "West"),
    ("missing_value",       "customer_id",       "2024-01-15 02:20:00", "orders_2.csv",  "NULL",              "West"),
    ("missing_value",       "customer_id",       "2024-01-15 02:20:00", "orders_3.csv",  "NULL",              "South"),
    ("invalid_reference",   "customer_id",       "2024-01-15 02:20:00", "orders_1.csv",  "XX-99999",          "East"),
    ("invalid_reference",   "customer_id",       "2024-01-15 02:20:00", "orders_1.csv",  "ZZ-00001",          "East"),
    ("invalid_reference",   "vendor_id",         "2024-01-15 02:20:00", "orders_2.csv",  "VEN99",             "Central"),
    ("invalid_date_format", "order_purchase_date","2024-01-15 02:20:00","orders_2.csv",  "32/13/2023",        "West"),
    ("invalid_date_format", "order_purchase_date","2024-01-15 02:20:00","orders_3.csv",  "2023-99-01",        "North"),
    ("invalid_status",      "order_status",      "2024-01-15 02:20:00", "orders_1.csv",  "UNKNWN",            "East"),
    ("invalid_status",      "order_status",      "2024-01-15 02:20:00", "orders_2.csv",  "pending_review",    "West"),
    ("missing_value",       "order_id",          "2024-01-15 02:20:00", "orders_3.csv",  "NULL",              "South"),
]

quarantine_orders_schema = StructType([
    StructField("issue_type",            StringType(), True),
    StructField("field_affected",        StringType(), True),
    StructField("quarantine_timestamp",  StringType(), True),
    StructField("source_file",           StringType(), True),
    StructField("raw_value",             StringType(), True),
    StructField("region",                StringType(), True),
])

df_qo = spark.createDataFrame(quarantine_orders_data, schema=quarantine_orders_schema)
df_qo.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
     .saveAsTable("dbx_retail.silver.quarantine_orders")

print(f"quarantine_orders : {df_qo.count()} rows")
display(df_qo)

In [0]:

quarantine_transactions_data = [
    ("negative_amount",   "profit",          "2024-01-15 02:30:00", "transactions_1.csv", "-999.99",  "West"),
    ("negative_amount",   "profit",          "2024-01-15 02:30:00", "transactions_1.csv", "-450.00",  "East"),
    ("negative_amount",   "Sales",           "2024-01-15 02:30:00", "transactions_2.csv", "-12.50",   "South"),
    ("negative_amount",   "Sales",           "2024-01-15 02:30:00", "transactions_3.csv", "-0.01",    "North"),
    ("invalid_format",    "discount",        "2024-01-15 02:30:00", "transactions_2.csv", "ABC%",     "West"),
    ("invalid_format",    "discount",        "2024-01-15 02:30:00", "transactions_2.csv", "200%",     "Central"),
    ("missing_value",     "Product_id",      "2024-01-15 02:30:00", "transactions_1.csv", "NULL",     "East"),
    ("missing_value",     "Product_id",      "2024-01-15 02:30:00", "transactions_3.csv", "NULL",     "West"),
    ("invalid_reference", "Order_id",        "2024-01-15 02:30:00", "transactions_2.csv", "XX-99999", "South"),
    ("zero_quantity",     "Quantity",        "2024-01-15 02:30:00", "transactions_3.csv", "0",        "East"),
    ("zero_quantity",     "Quantity",        "2024-01-15 02:30:00", "transactions_1.csv", "0",        "West"),
    ("zero_quantity",     "Quantity",        "2024-01-15 02:30:00", "transactions_3.csv", "0",        "Central"),
]

quarantine_transactions_schema = StructType([
    StructField("issue_type",            StringType(), True),
    StructField("field_affected",        StringType(), True),
    StructField("quarantine_timestamp",  StringType(), True),
    StructField("source_file",           StringType(), True),
    StructField("raw_value",             StringType(), True),
    StructField("region",                StringType(), True),
])

df_qt = spark.createDataFrame(quarantine_transactions_data, schema=quarantine_transactions_schema)
df_qt.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
     .saveAsTable("dbx_retail.silver.quarantine_transactions")

print(f"quarantine_transactions : {df_qt.count()} rows")
display(df_qt)


# =============================================================================
# GOLD LAYER — DIMENSIONS
# =============================================================================

In [0]:
# Based on real vendors.csv: VEN01-VEN07

dim_vendors_data = [
    ("VEN01", "Velocity Logistics"),
    ("VEN02", "Seaborne Ltd."),
    ("VEN03", "Alpine Supply Co."),
    ("VEN04", "Meridian Wholesale"),
    ("VEN05", "Crestview Distributors"),
    ("VEN06", "Hartwell Trading"),
    ("VEN07", "Pinnacle Goods"),
]

dim_vendors_schema = StructType([
    StructField("vendor_id",   StringType(), False),
    StructField("vendor_name", StringType(), True),
])

df_vendors = spark.createDataFrame(dim_vendors_data, schema=dim_vendors_schema)
df_vendors.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
          .saveAsTable("dbx_retail.gold.dim_vendors")

print(f"dim_vendors : {df_vendors.count()} rows")
display(df_vendors)

In [0]:
# Based on real customers_1.csv schema.
# Includes customers across 5 regions. Some customers have cross-region
# order history (used to test UC2 cross-region fraud rule).

dim_customers_data = [
    # customer_id, customer_email,             customer_name,       segment,       country,        city,           state,        postal_code, region
    ("CUST-001",  "adam.b@dbx_retail.com",     "Adam Bellavance",   "Home Office", "United States", "New York City","New York",    10009, "East"),
    ("CUST-002",  "sarah.k@dbx_retail.com",    "Sarah Kim",         "Corporate",   "United States", "Los Angeles",  "California",  90001, "West"),
    ("CUST-003",  "mike.j@dbx_retail.com",     "Mike Johnson",      "Consumer",    "United States", "Chicago",      "Illinois",    60601, "Central"),
    ("CUST-004",  "lisa.w@dbx_retail.com",     "Lisa Wong",         "Corporate",   "United States", "Houston",      "Texas",       77001, "South"),
    ("CUST-005",  "james.t@dbx_retail.com",    "James Taylor",      "Home Office", "United States", "Seattle",      "Washington",  98101, "West"),
    ("CUST-006",  "emma.d@dbx_retail.com",     "Emma Davis",        "Consumer",    "United States", "Miami",        "Florida",     33101, "South"),
    ("CUST-007",  "robert.m@dbx_retail.com",   "Robert Martinez",   "Corporate",   "United States", "Denver",       "Colorado",    80201, "Central"),
    ("CUST-008",  "olivia.s@dbx_retail.com",   "Olivia Smith",      "Consumer",    "United States", "Boston",       "Massachusetts",2101, "East"),
    ("CUST-009",  "william.b@dbx_retail.com",  "William Brown",     "Home Office", "United States", "Phoenix",      "Arizona",     85001, "West"),
    ("CUST-010",  "sophia.g@dbx_retail.com",   "Sophia Garcia",     "Corporate",   "United States", "Minneapolis",  "Minnesota",   55401, "North"),
    # Fraud candidates — high return volume and cross-region activity
    ("CUST-FR1",  "fraud.suspect1@mail.com",   "Alex Fraudster",    "Consumer",    "United States", "Las Vegas",    "Nevada",      89101, "West"),
    ("CUST-FR2",  "fraud.suspect2@mail.com",   "Jordan Anomaly",    "Home Office", "United States", "Portland",     "Oregon",      97201, "West"),
    ("CUST-FR3",  "fraud.suspect3@mail.com",   "Casey Crossregion", "Consumer",    "United States", "Atlanta",      "Georgia",     30301, "South"),
]

dim_customers_schema = StructType([
    StructField("customer_id",    StringType(),  False),
    StructField("customer_email", StringType(),  True),
    StructField("customer_name",  StringType(),  True),
    StructField("segment",        StringType(),  True),
    StructField("country",        StringType(),  True),
    StructField("city",           StringType(),  True),
    StructField("state",          StringType(),  True),
    StructField("postal_code",    IntegerType(), True),
    StructField("region",         StringType(),  True),
])

df_customers = spark.createDataFrame(dim_customers_data, schema=dim_customers_schema)
df_customers.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
            .saveAsTable("dbx_retail.gold.dim_customers")

print(f"dim_customers : {df_customers.count()} rows")
display(df_customers)

In [0]:
# Based on real products.json schema.
# product_id format matches transactions files: FUR-, OFF-, TEC- prefixes.

dim_products_data = [
    # product_id,              product_name,                         category,       brand,             unit_price, region,    vendor_id
    ("FUR-CH-10001453",  "Ergonomic Mesh Office Chair",              "Furniture",    "SitPro",          299.99, "East",    "VEN01"),
    ("FUR-TA-10001039",  "Adjustable Standing Desk",                 "Furniture",    "DeskMaster",      549.00, "West",    "VEN02"),
    ("FUR-BO-10001798",  "5-Shelf Bookcase Walnut",                  "Furniture",    "ShelfCo",          89.99, "Central", "VEN03"),
    ("OFF-PA-10001970",  "Premium Copy Paper 500 Sheet",             "Office",       "PaperMax",          9.99, "East",    "VEN01"),
    ("OFF-SU-10000618",  "Stapler Heavy Duty",                       "Office",       "BindRight",        24.99, "South",   "VEN04"),
    ("OFF-BI-10003527",  "3-Ring Binder 2 Inch",                     "Office",       "OfficeWorks",       7.49, "West",    "VEN02"),
    ("TEC-CO-10004115",  "Wireless Bluetooth Headset",               "Technology",   "SoundGear",       179.99, "West",    "VEN05"),
    ("TEC-PH-10002275",  "Smartphone 128GB Unlocked",                "Technology",   "TechBrand",       699.99, "East",    "VEN06"),
    ("TEC-AC-10003833",  "USB-C 65W Charging Hub",                   "Technology",   "PowerUp",          49.99, "North",   "VEN05"),
    ("TEC-MA-10000418",  "15.6 Laptop Business Grade",               "Technology",   "CompuBrand",     1199.00, "West",    "VEN07"),
    ("FUR-FU-10003274",  "Filing Cabinet 4-Drawer",                  "Furniture",    "StoreMate",       199.99, "South",   "VEN03"),
    ("OFF-EN-10002986",  "Security Envelopes Box 100",               "Office",       "MailPro",           8.99, "Central", "VEN04"),
    # Slow-moving products — will appear in mv_slow_moving_products
    ("FUR-CH-10004389",  "Antique Writing Desk Solid Oak",           "Furniture",    "HeritageFurn",    899.00, "West",    "VEN03"),
    ("TEC-CO-10003601",  "Fax Machine Thermal Print",                "Technology",   "OldTech",         299.00, "East",    "VEN06"),
    ("OFF-AP-10002311",  "Manual Typewriter Portable",               "Office",       "RetroOffice",     149.99, "Central", "VEN04"),
]

dim_products_schema = StructType([
    StructField("product_id",   StringType(), False),
    StructField("product_name", StringType(), True),
    StructField("category",     StringType(), True),
    StructField("brand",        StringType(), True),
    StructField("unit_price",   FloatType(),  True),
    StructField("region",       StringType(), True),
    StructField("vendor_id",    StringType(), True),
])

df_products = spark.createDataFrame(dim_products_data, schema=dim_products_schema)
df_products.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
           .saveAsTable("dbx_retail.gold.dim_products")

print(f"dim_products : {df_products.count()} rows")
display(df_products)


# =============================================================================
# GOLD LAYER — FACTS
# =============================================================================

In [0]:
# Based on real orders_1.csv schema.
# Includes orders for all customers across regions.
# Fraud candidates (CUST-FR1, FR2, FR3) have orders in multiple regions.

fact_orders_data = [
    # order_id,          customer_id,  vendor_id, ship_mode,      order_status, order_purchase_date,  region
    ("CA-2017-138422",  "CUST-001",  "VEN01", "First Class",    "delivered",  "2023-01-10 09:00:00", "East"),
    ("CA-2017-138423",  "CUST-001",  "VEN02", "Second Class",   "delivered",  "2023-02-14 11:30:00", "East"),
    ("CA-2017-138424",  "CUST-002",  "VEN01", "Standard Class", "delivered",  "2023-01-20 14:00:00", "West"),
    ("CA-2017-138425",  "CUST-002",  "VEN03", "First Class",    "delivered",  "2023-03-05 08:45:00", "West"),
    ("CA-2017-138426",  "CUST-003",  "VEN02", "Standard Class", "delivered",  "2023-01-25 16:20:00", "Central"),
    ("CA-2017-138427",  "CUST-004",  "VEN04", "Second Class",   "delivered",  "2023-02-08 10:10:00", "South"),
    ("CA-2017-138428",  "CUST-005",  "VEN05", "First Class",    "delivered",  "2023-01-30 13:00:00", "West"),
    ("CA-2017-138429",  "CUST-005",  "VEN05", "Standard Class", "delivered",  "2023-03-12 09:30:00", "West"),
    ("CA-2017-138430",  "CUST-006",  "VEN04", "Second Class",   "delivered",  "2023-02-20 15:00:00", "South"),
    ("CA-2017-138431",  "CUST-007",  "VEN03", "Standard Class", "delivered",  "2023-01-15 11:00:00", "Central"),
    ("CA-2017-138432",  "CUST-008",  "VEN01", "First Class",    "delivered",  "2023-02-28 14:30:00", "East"),
    ("CA-2017-138433",  "CUST-009",  "VEN06", "Second Class",   "delivered",  "2023-03-20 10:00:00", "West"),
    ("CA-2017-138434",  "CUST-010",  "VEN07", "Standard Class", "delivered",  "2023-01-05 08:00:00", "North"),
    ("CA-2017-138435",  "CUST-010",  "VEN07", "First Class",    "delivered",  "2023-03-01 16:00:00", "North"),
    # Fraud candidate FR1 — 2 legitimate orders, but will have 14 returns
    ("CA-FRAUD-001",   "CUST-FR1",  "VEN05", "Standard Class", "delivered",  "2023-01-12 10:00:00", "West"),
    ("CA-FRAUD-002",   "CUST-FR1",  "VEN06", "Second Class",   "delivered",  "2023-02-18 14:00:00", "West"),
    # Fraud candidate FR2 — 3 orders but 9 returns (high ratio)
    ("CA-FRAUD-010",   "CUST-FR2",  "VEN01", "First Class",    "delivered",  "2023-01-08 09:00:00", "West"),
    ("CA-FRAUD-011",   "CUST-FR2",  "VEN02", "Standard Class", "delivered",  "2023-02-14 11:00:00", "West"),
    ("CA-FRAUD-012",   "CUST-FR2",  "VEN03", "Second Class",   "delivered",  "2023-03-10 13:00:00", "East"),  # cross-region order
    # Fraud candidate FR3 — cross-region orders AND high returns
    ("CA-FRAUD-020",   "CUST-FR3",  "VEN04", "Standard Class", "delivered",  "2023-01-15 10:00:00", "South"),
    ("CA-FRAUD-021",   "CUST-FR3",  "VEN05", "First Class",    "delivered",  "2023-02-22 12:00:00", "East"),  # cross-region
    ("CA-FRAUD-022",   "CUST-FR3",  "VEN06", "Second Class",   "delivered",  "2023-03-18 15:00:00", "West"),  # cross-region
]

fact_orders_schema = StructType([
    StructField("order_id",            StringType(), False),
    StructField("customer_id",         StringType(), True),
    StructField("vendor_id",           StringType(), True),
    StructField("ship_mode",           StringType(), True),
    StructField("order_status",        StringType(), True),
    StructField("order_purchase_date", StringType(), True),
    StructField("region",              StringType(), True),
])

df_orders = spark.createDataFrame(fact_orders_data, schema=fact_orders_schema)
df_orders.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
         .saveAsTable("dbx_retail.gold.fact_orders")

print(f"fact_orders : {df_orders.count()} rows")
display(df_orders)

In [0]:
# Based on real returns_1.json schema: order_id, return_reason, return_date,
# refund_amount, return_status.
# customer_id added at Gold layer via join with fact_orders.
#
# FRAUD SIGNALS built into the data:
#   CUST-FR1 : 14 returns on 2 orders (700% ratio) — triggers rules 1,2,3,4
#   CUST-FR2 : 9 returns on 3 orders  (300% ratio) — triggers rules 1,2,3
#   CUST-FR3 : 7 returns cross-region              — triggers rules 1,3,5
#   CUST-FR1 : includes CA-FRAUD-999 (no matching order) — triggers rule 4

fact_returns_data = [
    # order_id,          customer_id,  return_reason,          return_date,  refund_amount, return_status
    # Normal customers — 1-2 returns each
    ("CA-2017-138422",  "CUST-001", "Not Satisfied",          "2023-02-01",  125.50, "Approved"),
    ("CA-2017-138424",  "CUST-002", "Defective Product",      "2023-02-10",   89.99, "Approved"),
    ("CA-2017-138427",  "CUST-004", "Wrong Delivery",         "2023-02-25",  210.00, "Approved"),
    ("CA-2017-138428",  "CUST-005", "Changed Mind",           "2023-02-20",  179.99, "Approved"),
    ("CA-2017-138432",  "CUST-008", "Late Delivery",          "2023-03-10",   45.00, "Approved"),
    ("CA-2017-138434",  "CUST-010", "Wrong Item Received",    "2023-01-20",  299.99, "Approved"),

    # CUST-FR1 — 14 returns (2 orders, 700% ratio, high refund value, unmatched order)
    ("CA-FRAUD-001",   "CUST-FR1", "Not Satisfied",           "2023-01-20",  195.00, "Approved"),
    ("CA-FRAUD-001",   "CUST-FR1", "Defective Product",       "2023-01-22",  195.00, "Approved"),
    ("CA-FRAUD-001",   "CUST-FR1", "Changed Mind",            "2023-01-25",  195.00, "Approved"),
    ("CA-FRAUD-001",   "CUST-FR1", "Wrong Item Received",     "2023-01-28",  195.00, "Approved"),
    ("CA-FRAUD-001",   "CUST-FR1", "Late Delivery",           "2023-02-01",  195.00, "Approved"),
    ("CA-FRAUD-001",   "CUST-FR1", "Not Satisfied",           "2023-02-05",  195.00, "Approved"),
    ("CA-FRAUD-002",   "CUST-FR1", "Defective Product",       "2023-02-20",  250.00, "Approved"),
    ("CA-FRAUD-002",   "CUST-FR1", "Wrong Delivery",          "2023-02-22",  250.00, "Approved"),
    ("CA-FRAUD-002",   "CUST-FR1", "Not Satisfied",           "2023-02-25",  250.00, "Approved"),
    ("CA-FRAUD-002",   "CUST-FR1", "Changed Mind",            "2023-02-27",  250.00, "Approved"),
    ("CA-FRAUD-002",   "CUST-FR1", "Wrong Item Received",     "2023-03-01",  250.00, "Approved"),
    ("CA-FRAUD-002",   "CUST-FR1", "Not Satisfied",           "2023-03-03",  250.00, "Approved"),
    ("CA-FRAUD-002",   "CUST-FR1", "Defective Product",       "2023-03-05",  250.00, "Approved"),
    # Rule 4 trigger — order CA-FRAUD-999 does NOT exist in fact_orders
    ("CA-FRAUD-999",   "CUST-FR1", "Defective Product",       "2023-03-10",  399.00, "Approved"),

    # CUST-FR2 — 9 returns on 3 orders (300% ratio, high refund value)
    ("CA-FRAUD-010",   "CUST-FR2", "Not Satisfied",           "2023-01-15",  180.00, "Approved"),
    ("CA-FRAUD-010",   "CUST-FR2", "Defective Product",       "2023-01-18",  180.00, "Approved"),
    ("CA-FRAUD-010",   "CUST-FR2", "Changed Mind",            "2023-01-22",  180.00, "Approved"),
    ("CA-FRAUD-011",   "CUST-FR2", "Wrong Item Received",     "2023-02-20",  200.00, "Approved"),
    ("CA-FRAUD-011",   "CUST-FR2", "Not Satisfied",           "2023-02-23",  200.00, "Approved"),
    ("CA-FRAUD-011",   "CUST-FR2", "Late Delivery",           "2023-02-26",  200.00, "Approved"),
    ("CA-FRAUD-012",   "CUST-FR2", "Defective Product",       "2023-03-12",  220.00, "Approved"),
    ("CA-FRAUD-012",   "CUST-FR2", "Not Satisfied",           "2023-03-14",  220.00, "Approved"),
    ("CA-FRAUD-012",   "CUST-FR2", "Wrong Delivery",          "2023-03-16",  220.00, "Approved"),

    # CUST-FR3 — 7 returns cross-region (orders in South, East, West)
    ("CA-FRAUD-020",   "CUST-FR3", "Not Satisfied",           "2023-01-25",  150.00, "Approved"),
    ("CA-FRAUD-020",   "CUST-FR3", "Defective Product",       "2023-01-28",  150.00, "Approved"),
    ("CA-FRAUD-021",   "CUST-FR3", "Changed Mind",            "2023-03-01",  310.00, "Approved"),
    ("CA-FRAUD-021",   "CUST-FR3", "Wrong Item Received",     "2023-03-04",  310.00, "Approved"),
    ("CA-FRAUD-022",   "CUST-FR3", "Not Satisfied",           "2023-03-22",  280.00, "Approved"),
    ("CA-FRAUD-022",   "CUST-FR3", "Defective Product",       "2023-03-24",  280.00, "Approved"),
    ("CA-FRAUD-022",   "CUST-FR3", "Late Delivery",           "2023-03-26",  280.00, "Approved"),
]

fact_returns_schema = StructType([
    StructField("order_id",      StringType(), False),
    StructField("customer_id",   StringType(), True),
    StructField("return_reason", StringType(), True),
    StructField("return_date",   StringType(), True),
    StructField("refund_amount", FloatType(),  True),
    StructField("return_status", StringType(), True),
])

df_returns = spark.createDataFrame(fact_returns_data, schema=fact_returns_schema)
df_returns.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
          .saveAsTable("dbx_retail.gold.fact_returns")

print(f"fact_returns : {df_returns.count()} rows")
display(df_returns)


# =============================================================================
# GOLD LAYER — MATERIALIZED VIEWS (aggregations)
# =============================================================================

In [0]:
# Monthly revenue by region — built from fact_orders + transactions.
# Two months of data so UC4 can compute MoM delta.

mv_revenue_data = [
    # region,     order_month,  total_sales,  order_count
    ("East",    "2023-01",    42500.75,  320),
    ("West",    "2023-01",    61200.50,  410),
    ("Central", "2023-01",    28900.00,  215),
    ("South",   "2023-01",    33750.25,  280),
    ("North",   "2023-01",    19800.00,  145),
    # Month 2 — East down, West up, Central up, South flat, North up
    ("East",    "2023-02",    38200.00,  290),
    ("West",    "2023-02",    68500.00,  445),
    ("Central", "2023-02",    32100.50,  240),
    ("South",   "2023-02",    33200.00,  275),
    ("North",   "2023-02",    24100.00,  175),
]

mv_revenue_schema = StructType([
    StructField("region",      StringType(), True),
    StructField("order_month", StringType(), True),
    StructField("total_sales", FloatType(),  True),
    StructField("order_count", IntegerType(),True),
])

df_mv_revenue = spark.createDataFrame(mv_revenue_data, schema=mv_revenue_schema)
df_mv_revenue.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
             .saveAsTable("dbx_retail.gold.mv_revenue_by_region")

print(f"mv_revenue_by_region : {df_mv_revenue.count()} rows")
display(df_mv_revenue)

In [0]:
from pyspark.sql.functions import round as spark_round, col
from pyspark.sql.types import StructType, StructField, StringType, FloatType, IntegerType

# Return rate per vendor — VEN06 and VEN07 are high-risk (>20%)

mv_vendor_rr_data = [
    # vendor_id, vendor_name,             return_rate, total_returns, total_orders
    ("VEN01", "Velocity Logistics",        0.08,   42,  525),
    ("VEN02", "Seaborne Ltd.",             0.12,   58,  483),
    ("VEN03", "Alpine Supply Co.",         0.15,   71,  473),
    ("VEN04", "Meridian Wholesale",        0.18,   63,  350),
    ("VEN05", "Crestview Distributors",    0.22,   88,  400),   # above 20% threshold
    ("VEN06", "Hartwell Trading",          0.31,  124,  400),   # high risk
    ("VEN07", "Pinnacle Goods",            0.27,   94,  348),   # high risk
]

mv_vendor_rr_schema = StructType([
    StructField("vendor_id",      StringType(), True),
    StructField("vendor_name",    StringType(), True),
    StructField("return_rate",    FloatType(),  True),
    StructField("total_returns",  IntegerType(),True),
    StructField("total_orders",   IntegerType(),True),
])

df_mv_vendor = spark.createDataFrame(mv_vendor_rr_data, schema=mv_vendor_rr_schema) \
    .withColumn("return_rate_pct", spark_round(col("return_rate") * 100, 1))

df_mv_vendor.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("dbx_retail.gold.mv_return_rate_by_vendor")

print(f"mv_return_rate_by_vendor : {df_mv_vendor.count()} rows")
display(spark.table("dbx_retail.gold.mv_return_rate_by_vendor"))

In [0]:
# Products flagged as slow-moving by region.
# 3 slow-moving products covering different regions.

mv_slow_data = [
    # product_id,          product_name,                    category,     region,    is_slow_moving, unit_price, days_since_last_sale
    ("FUR-CH-10004389",  "Antique Writing Desk Solid Oak", "Furniture",  "West",    "Yes",  899.00,  95),
    ("TEC-CO-10003601",  "Fax Machine Thermal Print",      "Technology", "East",    "Yes",  299.00,  120),
    ("OFF-AP-10002311",  "Manual Typewriter Portable",     "Office",     "Central", "Yes",  149.99,  85),
    # Normal velocity products for contrast
    ("FUR-CH-10001453",  "Ergonomic Mesh Office Chair",    "Furniture",  "East",    "No",   299.99,  3),
    ("TEC-PH-10002275",  "Smartphone 128GB Unlocked",      "Technology", "West",    "No",   699.99,  1),
    ("OFF-PA-10001970",  "Premium Copy Paper 500 Sheet",   "Office",     "East",    "No",     9.99,  2),
]

mv_slow_schema = StructType([
    StructField("product_id",          StringType(), True),
    StructField("product_name",        StringType(), True),
    StructField("category",            StringType(), True),
    StructField("region",              StringType(), True),
    StructField("is_slow_moving",      StringType(), True),
    StructField("unit_price",          FloatType(),  True),
    StructField("days_since_last_sale",IntegerType(),True),
])

df_mv_slow = spark.createDataFrame(mv_slow_data, schema=mv_slow_schema)
df_mv_slow.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
          .saveAsTable("dbx_retail.gold.mv_slow_moving_products")

print(f"mv_slow_moving_products : {df_mv_slow.count()} rows")
display(df_mv_slow)


# =============================================================================
# VERIFICATION — Confirm all tables exist with correct row counts
# =============================================================================

In [0]:

print("\n" + "=" * 60)
print("VERIFICATION — ALL TABLES")
print("=" * 60)

tables = {
    "SILVER": [
        "dbx_retail.silver.quarantine_customers",
        "dbx_retail.silver.quarantine_orders",
        "dbx_retail.silver.quarantine_transactions",
    ],
    "GOLD — Dimensions": [
        "dbx_retail.gold.dim_vendors",
        "dbx_retail.gold.dim_customers",
        "dbx_retail.gold.dim_products",
    ],
    "GOLD — Facts": [
        "dbx_retail.gold.fact_orders",
        "dbx_retail.gold.fact_returns",
    ],
    "GOLD — Materialized Views": [
        "dbx_retail.gold.mv_revenue_by_region",
        "dbx_retail.gold.mv_return_rate_by_vendor",
        "dbx_retail.gold.mv_slow_moving_products",
    ]
}

all_pass = True
for layer, table_list in tables.items():
    print(f"\n{layer}:")
    for tbl in table_list:
        try:
            count = spark.table(tbl).count()
            status = "OK" if count > 0 else "EMPTY"
            if count == 0:
                all_pass = False
            print(f"  [{status}]  {tbl:<55} {count:>4} rows")
        except Exception as e:
            all_pass = False
            print(f"  [FAIL] {tbl:<55} ERROR: {e}")

print("\n" + "=" * 60)
print(f"OVERALL STATUS : {'ALL TABLES READY' if all_pass else 'CHECK FAILURES ABOVE'}")
print("=" * 60)

In [0]:
# Confirms the fraud signals are correctly embedded in the data.
# Expected:
#   CUST-FR1 : 14 returns, 2 orders, ratio=7.00, unmatched=1
#   CUST-FR2 :  9 returns, 3 orders, ratio=3.00
#   CUST-FR3 :  7 returns, 3 orders, cross-region

print("\nFRAUD DATA SPOT-CHECK:")
print("-" * 60)
spark.sql("""
    SELECT
        r.customer_id,
        COUNT(r.order_id)                            AS total_returns,
        ROUND(SUM(r.refund_amount), 2)               AS total_refund_value,
        COUNT(DISTINCT o.region)                     AS distinct_regions,
        SUM(CASE WHEN o.order_id IS NULL THEN 1 ELSE 0 END) AS unmatched_returns
    FROM dbx_retail.gold.fact_returns r
    LEFT JOIN dbx_retail.gold.fact_orders o
        ON r.order_id = o.order_id
    WHERE r.customer_id IN ('CUST-FR1', 'CUST-FR2', 'CUST-FR3')
    GROUP BY r.customer_id
    ORDER BY total_returns DESC
""").show()

print("\nExpected results:")
print("  CUST-FR1 : 14 returns | high refund value | 1 unmatched order")
print("  CUST-FR2 :  9 returns | high refund value | 0 unmatched orders")
print("  CUST-FR3 :  7 returns | cross-region (South + East + West)")